In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["XLA_FLAGS"] = (
    "--xla_gpu_enable_command_buffer= "
    "--xla_gpu_strict_conv_algorithm_picker=false "
    "--xla_gpu_triton_gemm_any=true "
    "--xla_gpu_enable_latency_hiding_scheduler=true "
    "--xla_gpu_enable_highest_priority_async_stream=true "
)

from conformer.tokenizer import Tokenizer, HuggingFaceBPETokenizer
from conformer.config import (
    DataConfig,
    TrainingConfig,
    ConformerConfig,
    FeaturizerConfig,
)
from conformer.model import ZipformerEncoder
from flax import nnx
import optax
import grain
from conformer.dataset import batch_fn, ProcessAudioData
import jax
import jax.numpy as jnp
import jax.profiler
from pathlib import Path
import functools

# Enable JAX compilation caching
cache_dir = Path.home() / ".cache" / "jax_compilation_cache"
cache_dir.mkdir(parents=True, exist_ok=True)
jax.config.update("jax_compilation_cache_dir", str(cache_dir))

from tqdm.auto import tqdm

data_config = DataConfig()
train_config = TrainingConfig()

tokenizer_path = Path(data_config.tokenizer_path)
if tokenizer_path.suffix == ".json":
    tokenizer = HuggingFaceBPETokenizer.from_pretrained(tokenizer_path.parent)
else:
    tokenizer = Tokenizer.load_tokenizer(tokenizer_path)

featurizer_config = FeaturizerConfig()
conformer_config = ConformerConfig()
token_count = (
    tokenizer.vocab_size
    if hasattr(tokenizer, "vocab_size")
    else len(tokenizer.id_to_char)
)

model = ZipformerEncoder(
    token_count=token_count,
    num_layers=conformer_config.num_encoder_layers,
    d_model=conformer_config.encoder_dim,
    num_head=conformer_config.num_attention_heads,
    dropout=conformer_config.feed_forward_dropout_p,
    feed_forward_expansion_factor=conformer_config.feed_forward_expansion_factor,
    d_input=featurizer_config.n_mels,
    sample_rate=featurizer_config.sampling_rate,
    n_fft=featurizer_config.n_fft,
    n_window_size=featurizer_config.win_length,
    n_window_stride=featurizer_config.hop_length,
    dtype=jnp.bfloat16,
    rngs=nnx.Rngs(0),
)
model.initialize_weights(jax.random.PRNGKey(0))

W0306 14:51:22.359827   52155 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0306 14:51:22.362036   52027 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


In [2]:
nnx.display(model)